# ASC Income Dataset PoC

Testing HE on a subset of the [ASC Income Datset](https://fairlearn.org/main/user_guide/datasets/acs_income.html).

Please follow the setup suggested in the README.md file before running this code. 

## Installation

In [ ]:
import pandas as pd
from ingestor import download_and_clean_ascincome

## Ingestion

In [ ]:
df = download_and_clean_ascincome(states=['MA'], 
                             year=2018, 
                             horizon='1-Year', 
                             features='all', 
                             min_age=18)

In [ ]:
# Processing for HE: drop unwanted columns and one-hot encode categorical features
df_processed = df[["AGEP", "COW", "SCHL", "MAR", "RELP", "WKHP", "SEX", "RAC1P", "PINCP", "PWGTP", "income_over_50k"]]
df_processed = pd.get_dummies(df_processed, columns=["COW", "SCHL", "MAR", "RELP", "SEX", "RAC1P"], dtype=int)
df_processed.head()

In [ ]:
# Drop columns with NA and randomly select 1000 rows for testing
df_processed = df_processed.dropna()
df_sampled = df_processed.sample(n=1000, random_state=42)

In [ ]:
df_sampled.describe()

## Phase one: feature generation on raw vs. encrypted data

Experiment on the 1,000-row subsample.

In [ ]:
import numpy as np
from he_pipeline import (DataOwner, UntrustedResearcher, Measure, compare,
                         features_plain, package_nbytes)

df_raw_sample = df.loc[df_sampled.index]

costs = {}

### Step 1: Generate features on the raw data

In [ ]:
with Measure() as m:
    features_raw = features_plain(df_raw_sample)
costs["1. raw features"] = m.summary()
features_raw

### Step 2: Encrypt the dataset (Trusted Data Owner)


In [ ]:
with Measure() as m:
    owner = DataOwner()
    encoded = owner.encode(df_raw_sample)
    package = owner.encrypt(encoded)
    public_context = owner.public_context()
costs["2. encrypt"] = m.summary()

plain_bytes = encoded.to_numpy().nbytes
cipher_bytes = package_nbytes(package)
print(f"Encrypted columns: {len(package['columns'])}, rows: {package['n_rows']}")
print(f"Plaintext size (float64): {plain_bytes / 2**10:,.1f} KiB")
print(f"Ciphertext size: {cipher_bytes / 2**20:,.1f} MiB")
print(f"Ciphertext expansion: {cipher_bytes / plain_bytes:,.1f}x")
print(f"Public context size: {len(public_context) / 2**20:,.1f} MiB")

### Step 3: Generate features on the encrypted dataset (Untrusted Researcher)

In [ ]:
with Measure() as m:
    researcher = UntrustedResearcher(public_context)
    encrypted_stats = researcher.compute(package)
costs["3. encrypted compute"] = m.summary()

print(f"Encrypted statistics returned: {len(encrypted_stats)}")
print(f"Size of returned ciphertexts: {sum(map(len, encrypted_stats.values())) / 2**20:,.1f} MiB")

### Step 4: Decrypt to get the features (Trusted Data Owner)

In [ ]:
with Measure() as m:
    decrypted_stats = owner.decrypt(encrypted_stats)
    features_he = owner.finalize(decrypted_stats, package["n_rows"])
costs["4. decrypt + finalize"] = m.summary()
features_he

## Comparison: raw vs. HE

Numerical fidelity per feature type (RMSE, max absolute and max relative error). Empty groups give `NaN` in both pipelines and are excluded.

In [ ]:
comparison = compare(features_raw, features_he)
valid = comparison.dropna(subset=["value_raw", "value_he"])
fidelity = valid.groupby(["stat", "target"]).agg(
    n_features=("abs_err", "size"),
    rmse=("abs_err", lambda e: np.sqrt(np.mean(e ** 2))),
    max_abs_err=("abs_err", "max"),
    max_rel_err=("rel_err", "max"),
)
fidelity

In [ ]:
# Side-by-side view of the check-in features
view = comparison.set_index(["stat", "target", "group_var", "group"])[["value_raw", "value_he", "abs_err"]]
display(view.loc["count"].query("value_raw > 0"))
display(view.loc["mean"].loc[(slice(None), ["ALL", "INCOME_BRACKET", "SEX"]), :])
display(view.loc["corr"])

In [ ]:
cost_table = pd.DataFrame(costs).T
he_steps = ["2. encrypt", "3. encrypted compute", "4. decrypt + finalize"]
print(f"HE pipeline latency: {cost_table.loc[he_steps, 'latency_s'].sum():.2f} s "
      f"({cost_table.loc[he_steps, 'latency_s'].sum() / cost_table.loc['1. raw features', 'latency_s']:,.0f}x raw)")
cost_table